# Advanced Polygenic Risk Scores (PRS)

> **Important notes for this notebook**
>
> In `GWAS6` and `GWAS7`, we built polygenic scores (PRS) with
> **clumping and thresholding (C+T)** (`PLINK` and `PRSice-2`): keep one
> SNP per LD region, keep the SNPs below a $p$-value threshold, and use
> their GWAS effect sizes unchanged. In this notebook, we use methods
> that fit **all SNPs together** and **shrink** noisy effect estimates
> towards zero, using the SNP heritability from `GWAS8`. We compare two
> approaches, both in `LDAK`: 1. **From summary statistics**:
> **`MegaPRS`** (`ldak --mega-prs`), similar to `LDpred2` and `PRS-CS`
> 2. **From individual-level data**: **`Bolt-Predict`** (`ldak --bolt`)
> and **`Elastic-Predict`** (`ldak --elastic`), trained on the simulated
> cohort from `GWAS8`/`GWAS9` (`Data/SIM.QC`, $N = 5,000$)
>
> ## Learning outcomes
>
> - **Contrast** C+T (`PRSice-2`) with methods that fit all SNPs jointly
>   and shrink their effects (`MegaPRS`, `LDpred2`, `PRS-CS`,
>   `Bolt-Predict`, `Elastic-Predict`)
> - **Explain** how the expected heritability of each SNP
>   ($\mathbb{E}[h_j^2]$) controls how much its effect is shrunk
> - **Build** a polygenic score from summary statistics with `MegaPRS`
>   (`ldak --mega-prs`)
> - **Build** polygenic scores from individual-level data with
>   `ldak --bolt` and `ldak --elastic`, and evaluate them with the
>   incremental $R^2$ in `R`
>
> ## How to make this notebook work
>
> - In this notebook, we will use both `bash command line` and `R`
>   programming languages. Remember to **change the kernel** whenever
>   you transition from one language to the other
>   (`Kernel --> Change Kernel`) indicated by the languages’ images. We
>   will first run `Bash` commands.
>
> <img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash
> kernel

> **Some theory**
>
> ### Intuition First: From an “On/Off Switch” (`PRSice-2`) to a “Volume Knob” (Shrinkage)
>
> C+T (`PRSice-2`, `GWAS6` and `GWAS7`) works like an **on/off
> switch**: - **Clumping** keeps only the top SNP in each LD region and
> discards the rest, even if the region has more than one causal variant
> (`GWAS10`). - **Thresholding** uses a SNP’s GWAS effect
> $\hat{\beta}_j$ at full size if $p_j < P_T$, and ignores it (effect 0)
> otherwise.
>
> Shrinkage methods (`MegaPRS`, `LDpred2`, `PRS-CS`, `Bolt-Predict`,
> `Elastic-Predict`) work like a **volume knob**: they keep all SNPs,
> account for the LD between them, and turn each effect down according
> to how much we trust it. SNPs that are expected to explain little
> heritability ($h_j^2$ from `GWAS8`) and SNPs with noisy estimates are
> turned down the most.
>
> ### Why Move Beyond Clumping + Thresholding (C+T)?
>
> A polygenic score for individual $i$ is
> $\text{PRS}_i = \sum_{j=1}^M X_{ij} \tilde{\beta}_j$, where
> $\tilde{\beta}_j$ is the weight of SNP $j$. Shrinkage methods estimate
> all weights **jointly**. In the simplest case, the **ridge** model
> (each SNP effect is Gaussian with variance $h_j^2$; standardized
> genotypes and phenotype), the weights are
>
> $$\tilde{\beta} = \left( R + \frac{1}{N} \text{diag}\!\left(\frac{1}{h_j^2}\right) \right)^{-1} \hat{\beta}_{\text{GWAS}}$$
>
> where $R$ is the LD matrix and $N$ the GWAS sample size. Two things
> happen: multiplying by $R^{-1}$ removes the double counting of SNPs in
> LD, and the term $1/(N h_j^2)$ shrinks the effects. The shrinkage is
> strong when $N$ is small (noisy estimates) or $h_j^2$ is small (SNP
> not expected to matter).
>
> ### Summary-Statistics vs. Individual-Level PRS Estimators
>
> | Data Used | Method | Prior on SNP Effects ($b_j$) | Heritability Model | Input Data Required |
> |---------------|---------------|---------------|---------------|---------------|
> | **Summary statistics** | **C+T (`PRSice-2`)** | None: $p$-value threshold + LD clumping | None | GWAS summary statistics + LD reference |
> | **Summary statistics** | **`LDpred2`** | Point-normal: a fraction $p$ of SNPs has a Gaussian effect, the rest exactly 0 | GCTA ($\alpha = -1$) | GWAS summary statistics + LD matrix $R$ |
> | **Summary statistics** | **`PRS-CS`** | Continuous shrinkage prior (strong shrinkage for small effects, little for large ones) | GCTA ($\alpha = -1$) | GWAS summary statistics + pre-computed LD |
> | **Summary statistics** | **`MegaPRS`** (`--mega-prs`) | Choice of ridge (≈ `LDpred-inf`), bolt (two Gaussians, ≈ `LDpred2`), lasso or BayesR | LDAK ($\alpha = -0.25$) or BLD-LDAK | GWAS summary statistics + LD correlations (`ldak --calc-cors`) |
> | **Individual-level** | **`Bolt-Predict`** (`--bolt`) | Mixture of two Gaussians (large and small effects) | LDAK ($\alpha = -0.25$) | Genotypes ($X$) and phenotype ($y$) |
> | **Individual-level** | **`Elastic-Predict`** (`--elastic`) | Elastic net: mixture of a Laplace ($L_1$) and a Gaussian ($L_2$) prior | LDAK ($\alpha = -0.25$) | Genotypes ($X$) and phenotype ($y$) |
>
> **Which one should you use?** - **Summary statistics** (`MegaPRS`,
> `LDpred2`, `PRS-CS`): when the largest training data available is a
> published GWAS whose genotypes you cannot access. -
> **Individual-level** (`Bolt-Predict`, `Elastic-Predict`): when you
> have the genotypes and phenotypes of a large cohort yourself. Using
> $X$ and $y$ directly avoids the approximations of summary statistics
> and any mismatch with an LD reference panel ([Zhang et al.,
> 2021](https://doi.org/10.1038/s41467-021-24485-y)).
>
> > **Quick Concept Check (from Lecture 11)**: - *A trait is driven by a
> > few large-effect SNPs, while most SNPs have no effect. Which kind of
> > `MegaPRS` prior fits best?* A **sparse** prior such as
> > `--model bolt` (or `bayesr`), which shrinks most SNPs almost to zero
> > but leaves the large effects nearly untouched. For a highly
> > polygenic trait, `--model ridge` works as well. - *You have
> > genotypes of $N = 400$ people and a published GWAS of
> > $N = 250{,}000$. How should you train the PRS?* Use **`MegaPRS` on
> > the published summary statistics** with an LD reference panel, and
> > keep your 400 people for testing: the training sample size $N$ is
> > the main driver of PRS accuracy.

# Summary-Statistics Based PRS with `LDAK MegaPRS`

Let’s build a polygenic score for height from the GWAS summary
statistics (`Data/Height.QC.gz`) with **`MegaPRS`**: -
`--calc-cors <prefix>`: computes the LD correlations between SNPs within
1 Mb (`--window-kb 1000`) in the reference panel (`Data/EUR.QC`). -
`--mega-prs <prefix>`: estimates the SNP weights (`<prefix>.effects`)
from the summary statistics (`--summary`), the LD correlations
(`--cors`) and the per-SNP heritabilities from `GWAS8`
(`--ind-hers sumher_ldak.ind.hers`). With `--cv-proportion 0.1`, it
chooses the best model settings by **pseudo-validation**: it uses the
summary statistics of 90% of the GWAS sample to train and of the other
10% to test. The split is random, so we fix the random seed
(`--random-seed 42`) to make the results close to ours. -
`--model bolt`: a mixture of two Gaussians (the `LDAK` counterpart of
`LDpred2`). It tries several values of $p$ (the fraction of SNPs with
larger effects) and $f_2$ (the fraction of heritability they explain),
and keeps the best. - `--model ridge`: a single Gaussian for all SNPs
(the `LDAK` counterpart of `LDpred-inf`). - `--calc-scores <prefix>`:
multiplies each person’s genotypes by the SNP weights (`--scorefile`)
and adds them up; the scores are written to `<prefix>.profile`.
`--power 0` tells `LDAK` that the weights are per-allele effects that
should be used as they are.

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash
kernel.

Let’s link the `Data` folder, create `Results/GWAS11`, and run
`MegaPRS`:

In [ ]:
# Note: this step takes about 1 minute to run.
ln -sf ../Data
mkdir -p Results/GWAS11

# Prepare target phenotype and covariates for LDAK
awk 'NR>1 {print $1, $2, $3}' Data/EUR.height > Results/GWAS11/EUR.pheno
awk 'NR>1 {print $0}' Data/EUR.covariate > Results/GWAS11/EUR.covar

# Reuse Height.summaries and sumher_ldak.ind.hers from GWAS8 if present; otherwise create them
if [ -f Results/GWAS8/sumher_ldak.ind.hers ] && [ -f Results/GWAS8/Height.summaries ]; then
  cp Results/GWAS8/Height.summaries Results/GWAS11/Height.summaries
  cp Results/GWAS8/sumher_ldak.ind.hers Results/GWAS11/sumher_ldak.ind.hers
else
  gzip -dc Data/Height.QC.gz | awk '
  NR==1 {
    for (i=1; i<=NF; i++) col[$i]=i
    print "Predictor A1 A2 n Z"
    next
  }
  $col["MAF"] >= 0.01 && $col["INFO"] >= 0.8 && $col["SE"] > 0 && !seen[$col["SNP"]]++ {
    n = ("N" in col) ? $col["N"] : 388028
    beta = ("BETA" in col) ? $col["BETA"] : log($col["OR"])
    print $col["SNP"], $col["A1"], $col["A2"], n, beta / $col["SE"]
  }' > Results/GWAS11/Height.summaries
  ldak --thin Results/GWAS11/thin --bfile Data/EUR.QC --window-prune 0.98 --window-kb 100
  awk '{print $1, 1}' Results/GWAS11/thin.in > Results/GWAS11/weights.thin
  ldak --calc-tagging Results/GWAS11/tag_ldak --bfile Data/EUR.QC --weights Results/GWAS11/weights.thin --power -0.25 --window-kb 1000
  ldak --sum-hers Results/GWAS11/sumher_ldak --summary Results/GWAS11/Height.summaries --tagfile Results/GWAS11/tag_ldak.tagging --matrix Results/GWAS11/tag_ldak.matrix
fi

# 1. Compute local LD correlation windows (1000 kb) for MegaPRS
ldak \
  --calc-cors Results/GWAS11/cors_eur \
  --bfile Data/EUR.QC \
  --window-kb 1000

# 2. Run MegaPRS with Bolt (LDpred2 spike-and-slab) and Ridge (LDpred-inf) priors
ldak \
  --mega-prs Results/GWAS11/megaprs_bolt \
  --model bolt \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --summary Results/GWAS11/Height.summaries \
  --cors Results/GWAS11/cors_eur \
  --cv-proportion 0.1 \
  --random-seed 42

ldak \
  --mega-prs Results/GWAS11/megaprs_ridge \
  --model ridge \
  --ind-hers Results/GWAS11/sumher_ldak.ind.hers \
  --summary Results/GWAS11/Height.summaries \
  --cors Results/GWAS11/cors_eur \
  --cv-proportion 0.1 \
  --random-seed 42

# 3. Score target individuals in Data/EUR.QC
ldak --calc-scores Results/GWAS11/prs_megaprs_bolt --bfile Data/EUR.QC --scorefile Results/GWAS11/megaprs_bolt.effects --power 0 --pheno Results/GWAS11/EUR.pheno
ldak --calc-scores Results/GWAS11/prs_megaprs_ridge --bfile Data/EUR.QC --scorefile Results/GWAS11/megaprs_ridge.effects --power 0 --pheno Results/GWAS11/EUR.pheno

> **Pseudo-validation estimates**
>
> `ldak --mega-prs` prints the estimated accuracy (“approximate
> correlation”) of the best model; the values for all models are in
> `megaprs_bolt.cors`. In our run, the best `Bolt` model had an
> approximate correlation of about $1.3$, which is impossible for a real
> correlation, while its actual correlation with height in `Data/EUR.QC`
> (printed by `--calc-scores`) was only about $0.40$. Pseudo-validation
> depends on the LD reference panel, and ours has only ~480 individuals
> (who are also the people we score), so its estimates are unreliable.
> Always check a PRS in a real target sample.
>
> Also look at which `Bolt` model was chosen (`megaprs_bolt.best` and
> `megaprs_bolt.cors`). Pseudo-validation picked a sparse model
> ($p = 0.02$: 2% of the SNPs explain half of the heritability), so most
> SNPs are shrunk strongly towards zero. Height is highly polygenic, and
> the `Ridge` model, which gives every SNP a small effect, predicts
> height better (incremental $R^2 \approx 0.23$ vs. $0.16$ for `Bolt`).
> With a small LD panel, the “best” model by pseudo-validation is not
> necessarily the best model in reality.

# Individual-Level PRS with `LDAK-Bolt-Predict` and `LDAK-Elastic-Predict`

Now suppose we have the genotypes $X$ and phenotypes $y$ of a cohort
ourselves. With the ~480 individuals of `Data/EUR.QC`, individual-level
scores explain almost nothing (incremental $R^2 < 0.01$), so we use the
simulated cohort `Data/SIM.QC` ($N = 5,000$; true
$h^2_{\text{SNP}} = 0.5$). We: 1. Split it into a **training set** of
80% (`train.keep`, $N = 4,000$) and a **test set** of 20% (`test.keep`,
$N = 1,000$). 2. Estimate the SNP heritability in the training set
(`--reml`). 3. Train two prediction models on the training set.
`--LOCO NO` asks for one genome-wide model (`*.effects`) instead of 22
leave-one-chromosome-out models as in `GWAS9`: - `--bolt <prefix>`:
**Bolt-Predict** (mixture of two Gaussians), using the LDAK model
(`--power -0.25`). *(The per-SNP heritabilities `sumher_ldak.ind.hers`
from `GWAS8` belong to the height GWAS SNPs, so we cannot use them
here.)* - `--elastic <prefix>`: **Elastic-Predict** (elastic net; the
same model as Step 1 of `LDAK-KVIK`). 4. Compute the scores of the test
set with `ldak --calc-scores` (`--keep Results/GWAS11/test.keep`).

In [ ]:
# Note: this step takes about 2 minutes to run.
# Prepare the simulated phenotype (true h2_SNP = 0.5) and covariates for LDAK
awk 'NR>1 {print $1, $2, $3}' Data/SIM.pheno > Results/GWAS11/SIM.pheno
awk 'NR>1 {print $0}' Data/SIM.covariate > Results/GWAS11/SIM.covar

# 1. Create an 80% Training / 20% Testing split
awk 'NR % 5 != 0 {print $1, $2}' Results/GWAS11/SIM.pheno > Results/GWAS11/train.keep
awk 'NR % 5 == 0 {print $1, $2}' Results/GWAS11/SIM.pheno > Results/GWAS11/test.keep

# 2. Estimate SNP heritability on the 80% Training set using REML
ldak --calc-kins-direct Results/GWAS11/kin_train --bfile Data/SIM.QC --keep Results/GWAS11/train.keep --ignore-weights YES --power -0.25
ldak --reml Results/GWAS11/reml_train --grm Results/GWAS11/kin_train --pheno Results/GWAS11/SIM.pheno --covar Results/GWAS11/SIM.covar --keep Results/GWAS11/train.keep
grep Her_All Results/GWAS11/reml_train.reml

# 3. Fit Individual-Level Bolt-Predict (--bolt) and Elastic-Predict (--elastic) on Training set
ldak \
  --bolt Results/GWAS11/ind_bolt \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS11/SIM.pheno \
  --covar Results/GWAS11/SIM.covar \
  --keep Results/GWAS11/train.keep \
  --power -0.25 \
  --cv-proportion 0.1 \
  --random-seed 42 \
  --LOCO NO

ldak \
  --elastic Results/GWAS11/ind_elastic \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS11/SIM.pheno \
  --covar Results/GWAS11/SIM.covar \
  --keep Results/GWAS11/train.keep \
  --power -0.25 \
  --cv-proportion 0.1 \
  --random-seed 42 \
  --LOCO NO

# 4. Project trained weights onto the held-out 20% Test set
ldak --calc-scores Results/GWAS11/prs_ind_bolt --bfile Data/SIM.QC --keep Results/GWAS11/test.keep --scorefile Results/GWAS11/ind_bolt.effects --power 0 --pheno Results/GWAS11/SIM.pheno
ldak --calc-scores Results/GWAS11/prs_ind_elastic --bfile Data/SIM.QC --keep Results/GWAS11/test.keep --scorefile Results/GWAS11/ind_elastic.effects --power 0 --pheno Results/GWAS11/SIM.pheno

### Evaluating PRS Accuracy in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

As in `GWAS6` and `GWAS7`, we measure accuracy with the **incremental
$R^2$**: how much more of the phenotypic variance is explained when we
add the standardized score (`Profile_1`) to a model with only the
covariates (`Phenotype ~ Sex + PC1 + ... + PC6`). The `MegaPRS` scores
are evaluated for height in `Data/EUR.QC`, and the individual-level
scores for the simulated trait in the 1,000 test individuals of
`Data/SIM.QC`.

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 8.5, repr.plot.height = 4.5)

suppressMessages(suppressWarnings(library(ggplot2)))

# Helper function to compute incremental R2 over Sex + PC1..PC6
# Note: ldak --calc-scores outputs .profile with columns ID1, ID2, Phenotype, Covariates, Profile_1
calc_incremental_r2 <- function(profile_file, pheno_df, covar_df) {
  prs <- read.table(profile_file, header = TRUE)
  colnames(prs)[1:2] <- c("FID", "IID")
  colnames(pheno_df)[3] <- "Phenotype"
  dat <- merge(pheno_df, covar_df, by = c("FID", "IID"))
  dat <- merge(dat, prs[, c("FID", "IID", "Profile_1")], by = c("FID", "IID"))
  dat$PRS_std <- as.numeric(scale(dat$Profile_1))
  
  null_fit <- lm(Phenotype ~ Sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6, data = dat)
  full_fit <- lm(Phenotype ~ PRS_std + Sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6, data = dat)
  
  list(
    inc_r2 = summary(full_fit)$r.squared - summary(null_fit)$r.squared,
    p_val  = summary(full_fit)$coefficients["PRS_std", "Pr(>|t|)"],
    data   = dat
  )
}

pheno     <- read.table("Data/EUR.height", header = TRUE)
covar     <- read.table("Data/EUR.covariate", header = TRUE)
sim_pheno <- read.table("Data/SIM.pheno", header = TRUE)
sim_covar <- read.table("Data/SIM.covariate", header = TRUE)
n_train   <- length(readLines("Results/GWAS11/train.keep"))   # size of the individual-level training set
ind_label <- paste0("(SIM, N_train=", format(n_train, big.mark = ","), ")")

res_mega_bolt  <- calc_incremental_r2("Results/GWAS11/prs_megaprs_bolt.profile", pheno, covar)
res_mega_ridge <- calc_incremental_r2("Results/GWAS11/prs_megaprs_ridge.profile", pheno, covar)
res_ind_bolt   <- calc_incremental_r2("Results/GWAS11/prs_ind_bolt.profile", sim_pheno, sim_covar)
res_ind_elas   <- calc_incremental_r2("Results/GWAS11/prs_ind_elastic.profile", sim_pheno, sim_covar)

cat("Summary-Stats MegaPRS (Bolt / LDpred2) Incremental R2:     ", round(res_mega_bolt$inc_r2, 4), "(p =", signif(res_mega_bolt$p_val, 3), ")\n")
cat("Summary-Stats MegaPRS (Ridge / LDpred-inf) Inc. R2:        ", round(res_mega_ridge$inc_r2, 4), "(p =", signif(res_mega_ridge$p_val, 3), ")\n")
cat("Individual-Level LDAK-Bolt", ind_label, "Inc. R2:   ", round(res_ind_bolt$inc_r2, 4), "(p =", signif(res_ind_bolt$p_val, 3), ")\n")
cat("Individual-Level LDAK-Elastic", ind_label, "Inc. R2:", round(res_ind_elas$inc_r2, 4), "(p =", signif(res_ind_elas$p_val, 3), ")\n")

> **Stop - Read - Solve**
>
> - **Q1. Make a bar plot of the incremental $R^2$ (`inc_r2`) of the
>   four scores (`MegaPRS-Bolt`, `MegaPRS-Ridge`, `Individual-Bolt` and
>   `Individual-Elastic`).** *(The `MegaPRS` scores predict height and
>   the individual-level scores predict the simulated trait, so compare
>   the two pairs with care.)*
> - **Q2. The individual-level models were trained on 4,000 individuals
>   for a trait with $h^2_{\text{SNP}} = 0.5$. How close do they get to
>   this maximum? Trained on the ~380 individuals of `Data/EUR.QC`
>   instead, the same models explain less than 1% of the variance in
>   height (incremental $R^2 \approx 0.005$, $p > 0.4$). Why?** And why
>   do individual-level methods beat summary-statistics methods in the
>   UK Biobank ($N = 400{,}000$)?
>
> Hint: The expected accuracy of a score is roughly
> $R^2 \approx h^2_{\text{SNP}} \cdot \frac{N h^2_{\text{SNP}}}{N h^2_{\text{SNP}} + M_e}$,
> where $N$ is the training sample size and $M_e$ the number of
> independent effects that must be estimated. Compare $N = 388{,}028$
> (`Data/Height.QC.gz`), $N = 4,000$ (`Data/SIM.QC`) and $N \approx 380$
> (`Data/EUR.QC`).

In [ ]:
# Write your code here


> **Solution**
>
> In `R`, we put the incremental $R^2$ of the four scores in a table and
> plot them:
>
> ``` bash
> # Q1. Bar plot comparing incremental R2 across Summary-Stats and Individual-Level PRS models
> sum_cat <- "Summary Statistics (height, N=388k)"
> ind_cat <- paste0("Individual-Level (simulated trait, N=", format(n_train, big.mark = ","), ")")
> bench_df <- data.frame(
>   Method   = factor(c("MegaPRS (Bolt)", "MegaPRS (Ridge)", "LDAK-Bolt (Ind)", "LDAK-Elastic (Ind)"),
>                     levels = c("MegaPRS (Bolt)", "MegaPRS (Ridge)", "LDAK-Bolt (Ind)", "LDAK-Elastic (Ind)")),
>   Category = c(sum_cat, sum_cat, ind_cat, ind_cat),
>   Inc_R2   = c(res_mega_bolt$inc_r2, res_mega_ridge$inc_r2,
>                res_ind_bolt$inc_r2, res_ind_elas$inc_r2)
> )
>
> print(bench_df)
>
> p_prs <- ggplot(bench_df, aes(x = Method, y = Inc_R2, fill = Category)) +
>   geom_col(width = 0.6, col = "black") +
>   geom_text(aes(label = round(Inc_R2, 4)), vjust = -0.4, size = 4) +
>   scale_fill_manual(values = setNames(c("steelblue", "tomato"), c(sum_cat, ind_cat))) +
>   theme_bw(base_size = 13) +
>   ylim(0, max(bench_df$Inc_R2) * 1.18) +
>   labs(
>     title = "Incremental R2: Summary-Statistics MegaPRS vs. Individual-Level LDAK-Predict",
>     x = "PRS Estimator",
>     y = expression("Incremental " ~ R^2 ~ " over Covariates"),
>     fill = "Training Paradigm"
>   )
>
> show(p_prs)
> ```
>
> - **Q1**: The `MegaPRS` scores explain a substantial part of the
>   variance in height beyond the covariates: about 23% for `Ridge`
>   (incremental $R^2 \approx 0.23$) but only about 16% for `Bolt`
>   ($R^2 \approx 0.16$). Pseudo-validation chose a sparse `Bolt` model
>   (see the note above), which suits traits with some large effects;
>   height is highly polygenic, so the `Ridge` prior does better. The
>   individual-level `Bolt-Predict` and `Elastic-Predict` scores explain
>   about 13% of the variance of the simulated trait in the 1,000 test
>   individuals ($p < 10^{-30}$).
> - **Q2**: The individual-level scores reach about a quarter of the
>   maximum $h^2_{\text{SNP}} = 0.5$ ($R^2 \approx 0.13$; `reml_train`
>   estimates $h^2_{\text{SNP}} \approx 0.45$ in the training set).
>   Following the formula, accuracy depends on how $N h^2_{\text{SNP}}$
>   compares with $M_e$:
>   - With $N \approx 380$, $N h^2_{\text{SNP}}$ is only a few hundred,
>     far below $M_e$ (tens of thousands for genome-wide data), so the
>     estimated effects are mostly noise and are shrunk almost to zero.
>   - With $N = 4,000$ we do much better, partly because the simulated
>     trait is easy: only 1,000 causal SNPs, all of them among the
>     50,000 genotyped SNPs, which the sparse `Bolt`/`Elastic` priors
>     can find. Real traits such as height have many more causal
>     variants with tiny effects and need far larger samples.
>   - The height `MegaPRS` was trained on $N = 388{,}028$, so it is
>     limited less by $N$ than by the small LD reference panel and the
>     differences between the GWAS and target samples.
>   - When both approaches use the **same** large sample (as in the UK
>     Biobank), individual-level methods usually win, because they use
>     the exact LD of the sample instead of approximating it ([Zhang et
>     al., 2021](https://doi.org/10.1038/s41467-021-24485-y)).

------------------------------------------------------------------------

### Conclusion

In this tutorial, we brought the advanced part of the course together:
the **per-SNP heritabilities** from `GWAS8` (`sumher_ldak.ind.hers`),
the **GWAS summary statistics** (`Data/Height.QC.gz`) and the **LD
correlations** of a reference panel (`ldak --calc-cors`, the genome-wide
version of the LD matrix used in `GWAS10`) to build a polygenic score
with **`MegaPRS`**. We compared it with scores trained on
individual-level data (`ldak --bolt` and `ldak --elastic`; `--elastic`
is the same model as Step 1 of `LDAK-KVIK` in `GWAS9`).

> **Other take-home messages**
>
> - **Shrinkage beats clumping and thresholding**: `MegaPRS`, `LDpred2`
>   and `PRS-CS` keep correlated SNPs and shrink noisy GWAS estimates
>   using the SNP heritability, instead of switching SNPs on or off.
> - **Choose the method that fits your data**: use `ldak --mega-prs`
>   with large published GWAS summary statistics, and `ldak --bolt` or
>   `ldak --elastic` when you have the genotypes of a large cohort
>   yourself.
> - **Heritability links the whole advanced course**: we estimated
>   $h^2_{\text{SNP}}$ (`GWAS8`), used it to predict the genetic
>   background in mixed-model GWAS (`GWAS9`), met the same idea as the
>   effect-size prior in fine-mapping (`GWAS10`), and used it to shrink
>   polygenic score weights (`GWAS11`).